# Models & Tuning

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

train = pd.read_csv('train_fe.csv')
test = pd.read_csv('test_fe.csv')

train.head()

,PassengerId,HomePlanet,CryoSleep,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,...,GroupSize,IsAlone,TotalSpend,HasSpent_RoomService,HasSpent_FoodCourt,HasSpent_ShoppingMall,HasSpent_Spa,HasSpent_VRDeck,IsChild,Transported
0,0001_01,Europa,False,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,...,1,1,0.0,0,0,0,0,0,0,False
1,0002_01,Earth,False,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,...,1,1,736.0,1,1,1,1,1,0,True
2,0003_01,Europa,False,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,...,2,0,10383.0,1,1,0,1,1,0,False
3,0003_02,Europa,False,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,...,2,0,5176.0,0,1,1,1,1,0,False
4,0004_01,Earth,False,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,...,1,1,1091.0,1,1,1,1,1,0,True


### Признаки и OOF-функция

In [3]:
ALL_NUM = ['Age','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck','CabinNum','GroupSize','TotalSpend']
ALL_CAT = ['HomePlanet','CryoSleep','Destination','VIP','Deck','Side','IsAlone','IsChild',
           'HasSpent_RoomService','HasSpent_FoodCourt','HasSpent_ShoppingMall','HasSpent_Spa','HasSpent_VRDeck']
LOG_NUM = ['Age','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck','GroupSize']

y = train['Transported'].astype(int)
X_full = train[ALL_NUM + ALL_CAT].copy()
X_test_full = test[ALL_NUM + ALL_CAT].copy()
for c in ALL_CAT:
    X_full[c] = X_full[c].astype(str)
    X_test_full[c] = X_test_full[c].astype(str)

def cross_val_oof(fit_predict_fn, X, y, X_test, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))
    test_pred = np.zeros((n_splits, len(X_test)))
    for i, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        val_proba, test_proba = fit_predict_fn(X_tr, y_tr, X_val, X_test)
        oof[val_idx] = val_proba
        test_pred[i] = test_proba
    return oof, test_pred.mean(axis=0)

### LogisticRegression

In [4]:
def logreg_fit_predict(X_tr, y_tr, X_val, X_test):
    prep = ColumnTransformer([
        ('num', StandardScaler(), LOG_NUM),
        ('cat', OneHotEncoder(handle_unknown='ignore'), ALL_CAT),
    ])
    model = Pipeline([('prep', prep), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
    model.fit(X_tr, y_tr)
    return model.predict_proba(X_val)[:, 1], model.predict_proba(X_test)[:, 1]

oof_logreg, test_logreg = cross_val_oof(logreg_fit_predict, X_full, y, X_test_full)
accuracy_score(y, oof_logreg > 0.5)

0.7962728632232831

Совпадает с результатом из `03_feature_engineering` (0.7963) — та же модель и тот же набор признаков, теперь через общий OOF-фреймворк.

### XGBoost

In [5]:
def make_xgb_fit_predict(params):
    def fit_predict(X_tr, y_tr, X_val, X_test):
        Xtr, Xval, Xte = X_tr.copy(), X_val.copy(), X_test.copy()
        for c in ALL_CAT:
            Xtr[c] = Xtr[c].astype('category')
            Xval[c] = Xval[c].astype('category')
            Xte[c] = Xte[c].astype('category')
        model = XGBClassifier(**params, enable_categorical=True, tree_method='hist',
                               eval_metric='logloss', random_state=42)
        model.fit(Xtr, y_tr)
        return model.predict_proba(Xval)[:, 1], model.predict_proba(Xte)[:, 1]
    return fit_predict

oof_xgb_default, _ = cross_val_oof(make_xgb_fit_predict({}), X_full, y, X_test_full)
accuracy_score(y, oof_xgb_default > 0.5)

0.8038651788795582

С параметрами по умолчанию XGBoost немного обходит LogisticRegression: 0.7963 → 0.8046. Но рост в рамках погрешности, но модель становится менее интерпретируемой. 

In [6]:
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
    }
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    for tr_idx, val_idx in skf.split(X_full, y):
        Xtr, Xval = X_full.iloc[tr_idx].copy(), X_full.iloc[val_idx].copy()
        for c in ALL_CAT:
            Xtr[c] = Xtr[c].astype('category')
            Xval[c] = Xval[c].astype('category')
        model = XGBClassifier(**params, enable_categorical=True, tree_method='hist',
                               eval_metric='logloss', random_state=42)
        model.fit(Xtr, y.iloc[tr_idx])
        scores.append(accuracy_score(y.iloc[val_idx], model.predict(Xval)))
    return np.mean(scores)

study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(xgb_objective, n_trials=25)

study_xgb.best_value, study_xgb.best_params

(0.8114573836685167,
 {'n_estimators': 278,
  'max_depth': 4,
  'learning_rate': 0.05082341959721458,
  'subsample': 0.6563696899899051,
  'colsample_bytree': 0.9208787923016158,
  'min_child_weight': 1,
  'reg_lambda': 8.862326508576253})

In [7]:
oof_xgb, test_xgb = cross_val_oof(make_xgb_fit_predict(study_xgb.best_params), X_full, y, X_test_full)
accuracy_score(y, oof_xgb > 0.5)

0.81111238927873

Тюнинг (25 trials, Optuna, 3-fold внутри поиска) даёт 0.8046 → 0.8119 на честном 5-fold CV. Да, прирост есть, но он не особо впечатлительный с учетом резкого увеличния сложности модели, которую мы провели. 

### LightGBM

In [8]:
def make_lgbm_fit_predict(params):
    def fit_predict(X_tr, y_tr, X_val, X_test):
        Xtr, Xval, Xte = X_tr.copy(), X_val.copy(), X_test.copy()
        for c in ALL_CAT:
            Xtr[c] = Xtr[c].astype('category')
            Xval[c] = Xval[c].astype('category')
            Xte[c] = Xte[c].astype('category')
        model = LGBMClassifier(**params, random_state=42, verbosity=-1)
        model.fit(Xtr, y_tr, categorical_feature=ALL_CAT)
        return model.predict_proba(Xval)[:, 1], model.predict_proba(Xte)[:, 1]
    return fit_predict

oof_lgbm_default, _ = cross_val_oof(make_lgbm_fit_predict({}), X_full, y, X_test_full)
accuracy_score(y, oof_lgbm_default > 0.5)

0.8093868629932129

Default LightGBM: 0.8094 — чуть лучше default XGBoost.

In [9]:
def lgbm_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
    }
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    for tr_idx, val_idx in skf.split(X_full, y):
        Xtr, Xval = X_full.iloc[tr_idx].copy(), X_full.iloc[val_idx].copy()
        for c in ALL_CAT:
            Xtr[c] = Xtr[c].astype('category')
            Xval[c] = Xval[c].astype('category')
        model = LGBMClassifier(**params, random_state=42, verbosity=-1)
        model.fit(Xtr, y.iloc[tr_idx], categorical_feature=ALL_CAT)
        scores.append(accuracy_score(y.iloc[val_idx], model.predict(Xval)))
    return np.mean(scores)

study_lgbm = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_lgbm.optimize(lgbm_objective, n_trials=25)

study_lgbm.best_value, study_lgbm.best_params

(0.8124925009483248,
 {'n_estimators': 274,
  'max_depth': 8,
  'learning_rate': 0.03766126410140812,
  'num_leaves': 15,
  'subsample': 0.9124485526185495,
  'colsample_bytree': 0.7451239150812146,
  'min_child_samples': 24})

In [10]:
oof_lgbm, test_lgbm = cross_val_oof(make_lgbm_fit_predict(study_lgbm.best_params), X_full, y, X_test_full)
accuracy_score(y, oof_lgbm > 0.5)

0.8116875647072357

Тюнинг двигает LightGBM: 0.8094 → 0.8117. В общем не впечатляет.

### CatBoost

In [11]:
CAT_IDX = [X_full.columns.get_loc(c) for c in ALL_CAT]

def make_cb_fit_predict(params):
    def fit_predict(X_tr, y_tr, X_val, X_test):
        model = CatBoostClassifier(**params, random_seed=42, verbose=False, cat_features=CAT_IDX)
        model.fit(X_tr, y_tr)
        return model.predict_proba(X_val)[:, 1], model.predict_proba(X_test)[:, 1]
    return fit_predict

oof_cb_default, _ = cross_val_oof(make_cb_fit_predict({}), X_full, y, X_test_full)
accuracy_score(y, oof_cb_default > 0.5)

0.81111238927873

Default CatBoost сразу лучший среди дефолтных моделей: 0.8111. Учитывает категориальные признаки нативно, без OneHotEncoder.

In [14]:
def cb_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 500),
        'depth': trial.suggest_int('depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
    }
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    for tr_idx, val_idx in skf.split(X_full, y):
        model = CatBoostClassifier(**params, random_seed=42, verbose=False, cat_features=CAT_IDX)
        model.fit(X_full.iloc[tr_idx], y.iloc[tr_idx])
        scores.append(accuracy_score(y.iloc[val_idx], model.predict(X_full.iloc[val_idx])))
    return np.mean(scores)

study_cb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_cb.optimize(cb_objective, n_trials=15)

study_cb.best_value, study_cb.best_params

(0.8144483092105069,
 {'iterations': 496,
  'depth': 5,
  'learning_rate': 0.03777266149808222,
  'l2_leaf_reg': 2.2968214094943433})

In [15]:
oof_cb, test_cb = cross_val_oof(make_cb_fit_predict(study_cb.best_params), X_full, y, X_test_full)
accuracy_score(y, oof_cb > 0.5)

0.8126078453928448

После тюнинга CatBoost — лучшая одиночная модель: 0.8111 → 0.8126. Но приросты минимальные

### Сравнение моделей

In [16]:
pd.DataFrame({
    'model': ['LogisticRegression', 'XGBoost', 'LightGBM', 'CatBoost'],
    'cv_accuracy': [
        accuracy_score(y, oof_logreg > 0.5),
        accuracy_score(y, oof_xgb > 0.5),
        accuracy_score(y, oof_lgbm > 0.5),
        accuracy_score(y, oof_cb > 0.5),
    ],
}).sort_values('cv_accuracy', ascending=False)

,model,cv_accuracy
3,CatBoost,0.812608
2,LightGBM,0.811688
1,XGBoost,0.811112
0,LogisticRegression,0.796273


Все три бустинга сошлись в очень узкий диапазон (0.8117–0.8126), CatBoost минимально впереди. LogisticRegression незначительно отстаёт от бустингов, возможно там есть нелинейные взаимодействия 

### Сохранение OOF и test предсказаний для ансамбля

In [17]:
oof_df = pd.DataFrame({
    'PassengerId': train['PassengerId'],
    'Transported': y,
    'logreg': oof_logreg,
    'xgb': oof_xgb,
    'lgbm': oof_lgbm,
    'cb': oof_cb,
})

test_df = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'logreg': test_logreg,
    'xgb': test_xgb,
    'lgbm': test_lgbm,
    'cb': test_cb,
})

oof_df.to_csv('oof_predictions.csv', index=False)
test_df.to_csv('test_predictions.csv', index=False)

oof_df.shape, test_df.shape

((8693, 6), (4277, 5))